# NLP — evaluation and error analysis

Interactive inspection of the already executed baseline and learned-presence experiments. Existing metrics, predictions, split definitions, and audit statuses are read-only inputs; no evaluation report or prediction file is created here.

In [ ]:
from pathlib import Path
import csv
import json
import sys
from collections import Counter
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
RESULTS = ROOT / 'results/nlp'
def read_json(path): return json.loads(path.read_text(encoding='utf-8'))
def read_jsonl(path): return [json.loads(line) for line in path.open(encoding='utf-8') if line.strip()]
splits = read_json(ROOT / 'data/processed/nlp/splits.json')
print({name: len(ids) for name, ids in splits.items()})

## Existing baseline evaluation

In [ ]:
baseline_metrics = read_json(RESULTS / 'baseline/baseline_metrics_updated.json')
display(pd.json_normalize(baseline_metrics, sep='.').T.rename(columns={0: 'value'}).head(40))
baseline_predictions = read_jsonl(RESULTS / 'predictions/baseline_inferences_updated.jsonl')
prediction_summary = pd.DataFrame([{'records': len(baseline_predictions), 'has_lesion_true': sum(item['liver_lesions']['has_lesion'] is True for item in baseline_predictions), 'has_lesion_false': sum(item['liver_lesions']['has_lesion'] is False for item in baseline_predictions), 'abstained': sum(item['liver_lesions']['has_lesion'] is None for item in baseline_predictions)}])
display(prediction_summary)

## Frozen model metrics

In [ ]:
learned_metrics = read_json(RESULTS / 'final_test/learned_presence_test_metrics.json')
validation_metrics = read_json(RESULTS / 'learned_presence_validation/learned_presence_validation_metrics.json')
comparison = pd.DataFrame([
    {'experiment': 'learned_presence_validation', **{key: validation_metrics.get(key) for key in ['test_studies', 'evaluated', 'accuracy', 'precision', 'recall', 'f1', 'macro_f1']}},
    {'experiment': 'learned_presence_final_test', **{key: learned_metrics.get(key) for key in ['test_studies', 'evaluated', 'accuracy', 'precision', 'recall', 'f1', 'macro_f1']}},
])
display(comparison)

## Split and leakage context

In [ ]:
audit_path = ROOT / 'data/processed/nlp/target_consistency_audit.csv'
with audit_path.open(encoding='utf-8-sig', newline='') as handle:
    audit = pd.DataFrame(list(csv.DictReader(handle)))
display(audit['overall_status'].value_counts(dropna=False).rename('studies').to_frame())
split_rows = pd.DataFrame([(split, record_id) for split, ids in splits.items() for record_id in ids], columns=['split', 'id'])
display(split_rows.groupby('split').size().rename('studies').to_frame())
print({'cross_split_id_overlap': len(set(splits['train']) & set(splits['val']) | set(splits['train']) & set(splits['test']) | set(splits['val']) & set(splits['test'])), 'patient_level_leakage': 'not assessable: no patient identifier in NLP master records'})

## Inline error and uncertainty examples

In [ ]:
gt = {item['id']: item for item in read_jsonl(ROOT / 'data/processed/nlp/ground_truth.jsonl')}
errors = []
for item in baseline_predictions:
    truth = gt.get(item['id'], {}).get('ground_truth', {})
    prediction = item['liver_lesions'].get('has_lesion')
    if prediction is None or (truth and prediction != truth.get('has_lesion')):
        errors.append({'id': item['id'], 'source_label': truth.get('has_lesion'), 'prediction': prediction, 'category': 'abstention' if prediction is None else 'presence_error'})
display(pd.DataFrame(errors).head(20))
print({'abstentions_or_presence_errors': len(errors), 'baseline_records_inspected': len(baseline_predictions)})

## Summary

The notebook separates source-derived evaluation from clinical validation, preserves the frozen train/validation/test protocol, and makes abstentions and audit groups visible. It does not retrain models, use the test split for configuration, or write new result files.

In [ ]:
assert set(splits) == {'train', 'val', 'test'}
assert not (set(splits['train']) & set(splits['val']))
assert not (set(splits['train']) & set(splits['test']))
assert not (set(splits['val']) & set(splits['test']))
print('Evaluation inspection completed without writing files.')